# Opportunity Layer Construction

## Overview

This notebook uses processed POI category counts derived from the Ordnance Survey Points of Interest dataset to represent the relative availability of destination categories across Greater Manchester.

The notebook applies logarithmic transformation and normalisation to construct category-level opportunity distributions for context-dependent trip purposes. These distributions are subsequently integrated with the Behaviour Layer through the trip-purpose-to-POI ontology.

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |District_manchester_poi_counts|
| Source | Ordnance Survey - Points of Interest Documentation |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |District_manchester_poi_counts|
| Output File | |
| Notebook Author | Peter |
| Last Updated |2026-07-24|

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## 2. Set Up File Pathes

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
project_root = Path.cwd().parents[1]

input_file_path = project_root / "data" / "POI" / "processed" / "manchester" / "District_manchester_poi_counts.xlsx"

output_file_path = project_root / "output" / "opportunity_layer" / "opportunity_layer.json"

## 3. Load Processed POI Data

In [4]:
poi_counts = pd.read_excel(input_file_path)

print(f"Dataset loaded successfully.")
print(f"Number of POI categories: {len(poi_counts)}")

Dataset loaded successfully.
Number of POI categories: 10


## 4. Data Inspection

In [5]:
display(poi_counts.head(10))

,POI Category,POI Count
0,Accommodation,139
1,Education,618
2,Employment,6031
3,Food and Beverage,2731
4,Healthcare,942
5,Personal Services,1613
6,Public Services,2899
7,Recreation,1256
8,Retail,2856
9,Transport,3401


## 5. Data Processing

### 5.1 Logarithmic Transformation of POI Counts

The Opportunity Layer assumes that areas containing more POIs provide greater opportunities for activities. However, raw POI counts are highly skewed across categories, with certain categories (e.g., Employment or Retail) containing substantially more POIs than others.

To reduce the dominance of extremely large counts while preserving their relative ordering, a logarithmic transformation is applied using the natural logarithm:

\[
\log(1+x)
\]

where \(x\) is the original POI count.

The transformed values are subsequently used to construct opportunity distributions.

In [6]:
poi_counts_processed = poi_counts.copy()


def log_transform_poi_counts(poi_counts_df):
    """
    Apply log(1 + x) transformation to POI counts.
    """

    df = poi_counts_df.copy()

    df["Log Count"] = np.log1p(df["POI Count"])

    return df


poi_counts_processed = log_transform_poi_counts(poi_counts)

display(poi_counts_processed)

,POI Category,POI Count,Log Count
0,Accommodation,139,4.941642
1,Education,618,6.428105
2,Employment,6031,8.704834
3,Food and Beverage,2731,7.912789
4,Healthcare,942,6.849066
5,Personal Services,1613,7.386471
6,Public Services,2899,7.972466
7,Recreation,1256,7.136483
8,Retail,2856,7.957527
9,Transport,3401,8.132119


## 6. Opportunity Distribution Construction

### 6.1 Trip Purpose-to-POI Mapping

Trip purposes generated by the Behaviour Layer are first mapped to one or more candidate POI categories.

Most trip purposes correspond directly to a single POI category (e.g., Shopping → Retail). However, certain trip purposes, such as Personal Business, may reasonably occur across multiple POI categories.

The mapping below defines the candidate POI categories used when constructing opportunity distributions.

In [7]:
from ontology.TRIP_PURPOSE_TO_POI_MAPPING import TRIP_PURPOSE_TO_POI_MAPPING

TRIP_PURPOSE_TO_POI_MAPPING

,Shared Trip Purpose,Mapping Strategy,POI Categories
0,Commuting,Representative Mapping,[Employment]
1,Business,Representative Mapping,[Employment]
2,Visiting Friends Elsewhere,Representative Mapping,[Food and Beverage]
3,Entertainment or Public Activity,Representative Mapping,[Recreation]
4,Day Trip,Representative Mapping,[Recreation]
5,Just Walk,Representative Mapping,[Recreation]
6,Education,Direct Mapping,[Education]
7,Escort Education,Direct Mapping,[Education]
8,Shopping,Direct Mapping,[Retail]
9,Sport: Participate,Direct Mapping,[Recreation]


### 6.2 Construction of opportunity distributions

For each mapped trip purpose, the candidate POI categories are linked to their corresponding log-transformed POI counts. These values are then renormalised within each trip purpose to produce category-level allocation weights that sum to one.

For trip purposes associated with a single POI category, the resulting allocation weight is one. For context-dependent trip purposes associated with multiple candidate categories, the allocation is distributed according to their relative opportunity availability.

In [8]:
def build_opportunity_distribution(mapping_df, poi_counts_df):

    log_count_lookup = (
        poi_counts_df
        .set_index("POI Category")["Log Count"]
    )

    output_rows = []

    for _, row in mapping_df.iterrows():

        trip_purpose = row["Shared Trip Purpose"]
        strategy = row["Mapping Strategy"]
        categories = row["POI Categories"]

        if strategy == "Excluded Mapping":
            continue

        if strategy in {
            "Direct Mapping",
            "Representative Mapping"
        }:

            weights = [1.0]

        elif strategy == "Context-dependent Mapping":

            log_counts = log_count_lookup.loc[categories]

            weights = (
                log_counts / log_counts.sum()
            ).tolist()

        else:

            raise ValueError(
                f"Unknown mapping strategy: {strategy}"
            )

        for category, weight in zip(categories, weights):

            output_rows.append({
                "Shared Trip Purpose": trip_purpose,
                "Mapping Strategy": strategy,
                "POI Category": category,
                "Opportunity Weight": weight
            })

    return pd.DataFrame(output_rows)

The opportunity distribution is constructed by applying the mapping rules to each shared trip purpose.

In [9]:
opportunity_distribution = build_opportunity_distribution(
    mapping_df=TRIP_PURPOSE_TO_POI_MAPPING,
    poi_counts_df=poi_counts_processed
)

display(
    opportunity_distribution.style.format({
        "Opportunity Weight": "{:.4f}"
    })
)

,Shared Trip Purpose,Mapping Strategy,POI Category,Opportunity Weight
0,Commuting,Representative Mapping,Employment,1.0000
1,Business,Representative Mapping,Employment,1.0000
2,Visiting Friends Elsewhere,Representative Mapping,Food and Beverage,1.0000
3,Entertainment or Public Activity,Representative Mapping,Recreation,1.0000
4,Day Trip,Representative Mapping,Recreation,1.0000
5,Just Walk,Representative Mapping,Recreation,1.0000
6,Education,Direct Mapping,Education,1.0000
7,Escort Education,Direct Mapping,Education,1.0000
8,Shopping,Direct Mapping,Retail,1.0000
9,Sport: Participate,Direct Mapping,Recreation,1.0000


### 6.3 Validation

The resulting distributions are validated to ensure that all weights are within the interval \([0,1]\), sum to 1 within each trip purpose, and contain no duplicated trip-purpose-category combinations.

In [10]:
# All weights must be between 0 and 1
assert opportunity_distribution[
    "Opportunity Weight"
].between(0, 1).all()


# Weights must sum to 1 within each trip purpose
weight_sums = (
    opportunity_distribution
    .groupby("Shared Trip Purpose")["Opportunity Weight"]
    .sum()
)

assert np.allclose(
    weight_sums.to_numpy(),
    1.0
)


# Each trip-purpose-category combination must be unique
assert not opportunity_distribution.duplicated(
    subset=[
        "Shared Trip Purpose",
        "POI Category"
    ]
).any()


# Excluded mappings must not appear in the output
assert (
    opportunity_distribution["Mapping Strategy"]
    != "Excluded Mapping"
).all()


print("All opportunity distribution validation checks passed.")

All opportunity distribution validation checks passed.


## 7. Output Preparation

The validated opportunity distribution provides POI allocation weights for each retained shared trip purpose. Direct and representative mappings assign the full allocation weight to a single POI category, while context-dependent mappings distribute the allocation across multiple candidate categories according to their relative log-transformed POI availability.

Trip purposes classified as Excluded Mapping are not assigned to a POI category and are therefore omitted from the opportunity distribution. Visiting Friends at Private Home is retained in the ontology as a private-residence destination outside the scope of the POI classification.

In [11]:
from ontology.TRIP_PURPOSE_TO_POI_MAPPING import EXCLUDED_DESTINATION_OUTCOMES

EXCLUDED_DESTINATION_OUTCOMES

{'Visiting Friends at Private Home': 'Private Residence (Outside POI Scope)'}

In [12]:
# Construct the complete opportunity-layer output
opportunity_layer_output = {}

for _, mapping_row in TRIP_PURPOSE_TO_POI_MAPPING.iterrows():

    trip_purpose = mapping_row["Shared Trip Purpose"]
    mapping_strategy = mapping_row["Mapping Strategy"]

    purpose_distribution = opportunity_distribution[
        opportunity_distribution["Shared Trip Purpose"]
        == trip_purpose
    ]

    poi_distribution = {
        row["POI Category"]: float(row["Opportunity Weight"])
        for _, row in purpose_distribution.iterrows()
    }

    purpose_output = {
        "mapping_strategy": mapping_strategy,
        "poi_distribution": poi_distribution
    }

    # Record why an excluded purpose has no POI allocation
    if trip_purpose in EXCLUDED_DESTINATION_OUTCOMES:
        purpose_output["allocation_outcome"] = (
            EXCLUDED_DESTINATION_OUTCOMES[trip_purpose]
        )

    opportunity_layer_output[trip_purpose] = purpose_output


pd.DataFrame(opportunity_layer_output)

,Commuting,Business,Visiting Friends Elsewhere,Entertainment or Public Activity,Day Trip,Just Walk,Education,Escort Education,Shopping,Sport: Participate,Holiday: Base,Personal Business,Other Escort,Visiting Friends at Private Home
mapping_strategy,Representative Mapping,Representative Mapping,Representative Mapping,Representative Mapping,Representative Mapping,Representative Mapping,Direct Mapping,Direct Mapping,Direct Mapping,Direct Mapping,Direct Mapping,Context-dependent Mapping,Context-dependent Mapping,Excluded Mapping
poi_distribution,{'Employment': 1.0},{'Employment': 1.0},{'Food and Beverage': 1.0},{'Recreation': 1.0},{'Recreation': 1.0},{'Recreation': 1.0},{'Education': 1.0},{'Education': 1.0},{'Retail': 1.0},{'Recreation': 1.0},{'Accommodation': 1.0},"{'Healthcare': 0.149015490349917, 'Public Serv...","{'Employment': 0.1478872700618409, 'Retail': 0...",{}
allocation_outcome,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Private Residence (Outside POI Scope)


### 7.1 Output Validation

The complete opportunity-layer output was validated before export. The validation confirms that all trip purposes defined in the mapping ontology are retained, that POI allocation weights sum to one for every included trip purpose, and that excluded mappings contain no POI allocation. The specified allocation outcome for excluded destinations is also verified.

In [13]:
# Validate opportunity-layer output

expected_trip_purposes = set(
    TRIP_PURPOSE_TO_POI_MAPPING["Shared Trip Purpose"]
)

assert set(opportunity_layer_output) == expected_trip_purposes, (
    "Trip purposes in the output do not match the mapping ontology."
)

for trip_purpose, purpose_output in opportunity_layer_output.items():

    strategy = purpose_output["mapping_strategy"]
    poi_distribution = purpose_output["poi_distribution"]

    if strategy == "Excluded Mapping":

        assert (
            purpose_output["allocation_outcome"]
            == EXCLUDED_DESTINATION_OUTCOMES[trip_purpose]
        ), (
            f"{trip_purpose}: incorrect allocation outcome."
        )

        continue

    assert np.isclose(
        sum(poi_distribution.values()),
        1.0
    ), (
        f"{trip_purpose}: opportunity weights do not sum to 1."
    )

print("Opportunity-layer output validation passed.")

Opportunity-layer output validation passed.


## 8. Export Opportunity Layer

The validated opportunity-layer output is exported as a JSON lookup for subsequent framework integration. The file retains the mapping strategy and POI allocation distribution for all shared trip purposes, including the documented outcome of destinations outside the POI scope.

In [14]:
import json
from pathlib import Path

opportunity_output_path = output_file_path

with open(
    opportunity_output_path,
    "w",
    encoding="utf-8"
) as output_file:
    json.dump(
        opportunity_layer_output,
        output_file,
        indent=4,
        ensure_ascii=False
    )

print("Opportunity layer exported successfully.")
print(f"Output path: {opportunity_output_path}")

Opportunity layer exported successfully.
Output path: c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\opportunity_layer\opportunity_layer.json


### 8.1 Output Summary

The completed opportunity layer contains POI-category allocation information for 14 shared trip purposes. Thirteen trip purposes are assigned either single-category or context-dependent POI distributions, while Visiting Friends at Private Home is retained as an excluded mapping with no POI allocation because its destination lies outside the defined POI scope.

The exported JSON file provides the opportunity-layer lookup required for subsequent integration with the demographic and behavioural evidence layers.